| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |


**Final understanding:**

FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database

It manages:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


Your code was storing both documents and metadata along with FAISS, but the crucial point is:

FAISS was not storing them natively; LangChain's FAISS wrapper was storing them.

```
LangChain FAISS VectorStore
│
├── FAISS index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│         ├── page_content
│         └── metadata
│
└── index_to_docstore_id
      └── FAISS maps positions to Document IDs
```

In [ ]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In Chroma, documents, metadata, IDs, and embeddings are records within the same database collection: Chroma collection

In [ ]:
collection.add(
    ids=["chunk-1"],
    documents=["Llama 2 is a family of language models."],
    metadatas=[
        {
            "source": "llama2.pdf",
            "page": 5
        }
    ],
    embeddings=[[0.1, 0.2, 0.3]]
)

| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


https://www.trychroma.com/

In [8]:
! pip install -q langchain-google-genai langchain-openai langchain-community langchain-chroma langchain-text-splitters pypdf

In [3]:
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_1052/3536989817.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [21]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [9]:
# --------------------------------------------------
# 3. Load PDF
# --------------------------------------------------
file_path = "llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

Total pages: 77


In [10]:
# --------------------------------------------------
# 4. Create chunks
# --------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


In [12]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [ ]:
{
    "id": "doc-101",
    "embedding": [0.12, -0.45, 0.78, ...],
    "document": "Employees receive 20 days of annual leave.",
    "metadata": {
        "source": "hr_policy.pdf",
        "page": 5,
        "department": "HR"
    }
}

In [13]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)


Documents added: 175
Total documents stored: 175


In [14]:
# --------------------------------------------------
# 7. Create retriever
# --------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [15]:
# --------------------------------------------------
# 8. Test retriever
# --------------------------------------------------

query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)



--- Retrieved document 1 ---
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model Developers Meta AI
Variations Llama 2 comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model Architecture Llama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning
Metadata: {'source': 'llama2-research-paper.pdf', 'moddate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'producer': 'pdfTeX-1.40.25', 'trapped': '/False', 'creationdate': '2023-07-20T00:30:36+00:00', 'page': 76, 'total_pages': 77, 'title': '', 'page_label': '77', 'creator': 'LaTeX with hyperref', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'}

--- Retrieved document 2 ---
T

In [16]:
# --------------------------------------------------
# 9. Prompt
# --------------------------------------------------

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [17]:
# --------------------------------------------------
# 10. Format documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

In [22]:
# --------------------------------------------------
# 11. LLM
# --------------------------------------------------

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [24]:
# --------------------------------------------------
# 12. RAG chain
# --------------------------------------------------

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [25]:
# --------------------------------------------------
# 13. Ask question
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Final answer:

Based on the provided context, Llama 2 is an auto-regressive language model that uses an optimized standard transformer architecture. 

Its specific architectural details include:
* **Pre-normalization:** Uses RMSNorm.
* **Activation Function:** Uses the SwiGLU activation function.
* **Positional Embeddings:** Uses rotary positional embeddings (RoPE).
* **Key Enhancements over Llama 1:** Increased context length and Grouped-Query Attention (GQA).
* **Alignment (Tuned Versions):** Uses supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align to human preferences for helpfulness and safety.


In [26]:
# load vector store from a persistent directory
from langchain_chroma import Chroma

loaded_vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2"
)

In [27]:
loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [28]:
docs = loaded_retriever.invoke(
    "What is Llama 2?"
)

for doc in docs:
    print(doc.page_content[:500])

Feedback (RLHF). Although the training methodology is simple, high computational requirements have
limited the development of LLMs to a few players. There have been public releases of pretrained LLMs
(such as BLOOM (Scao et al., 2022), LLaMa-1 (Touvron et al., 2023), and Falcon (Penedo et al., 2023)) that
match the performance of closed pretrained competitors like GPT-3 (Brown et al., 2020) and Chinchilla
(Hoffmann et al., 2022), but none of these models are suitable substitutes for closed “prod
tion clusters for pretraining. Fine-tuning, annotation, and evaluation were also
performed on third-party cloud compute.
Carbon Footprint Pretraining utilized a cumulative 3.3M GPU hours of computation on hardware
of type A100-80GB (TDP of 350-400W). Estimated total emissions were 539
tCO2eq, 100% of which were offset by Meta’s sustainability program.
Training Data(Sections 2.1 and 3)
Overview Llama 2 was pretrained on 2 trillion tokens of data from publicly available
sources. The fine-tuning d

In [30]:
# vector_store.persist()

In [31]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "page": 10
        }
    }
)

In [32]:
results = vector_store.similarity_search(
    query="What is reinforcement learning?",
    k=5,
    filter={
        "page": 10
    }
)

In [34]:
len(results)

3

In [36]:
results[0].page_content

'Dataset\nNum. of\nComparisons\nAvg. # Turns\nper Dialogue\nAvg. # Tokens\nper Example\nAvg. # Tokens\nin Prompt\nAvg. # Tokens\nin Response\nAnthropic Helpful 122,387 3.0 251.5 17.7 88.4\nAnthropic Harmless 43,966 3.0 152.5 15.7 46.4\nOpenAI Summarize 176,625 1.0 371.1 336.0 35.1\nOpenAI WebGPT 13,333 1.0 237.2 48.3 188.9\nStackExchange 1,038,480 1.0 440.2 200.1 240.2\nStanford SHP 74,882 1.0 338.3 199.5 138.8\nSynthetic GPT-J 33,139 1.0 123.3 13.0 110.3\nMeta (Safety & Helpfulness) 1,418,091 3.9 798.5 31.4 234.1\nTotal 2,919,326 1.6 595.7 108.2 216.9\nTable 6: Statistics of human preference data for reward modeling.We list both the open-source and\ninternally collected human preference data used for reward modeling. Note that a binary human preference\ncomparison contains 2 responses (chosen and rejected) sharing the same prompt (and previous dialogue).\nEach example consists of a prompt (including previous dialogue if available) and a response, which is the\ninput of the reward mode